# COMMUN

### Imports et configuration

In [3]:
import os
import yaml
import pandas as pd
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path

### Charger variables d'environnement depuis .env

In [4]:
load_dotenv()

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5


True

In [5]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
print(CONFIG_PATH)

/home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/config.yml


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [4]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [5]:
test = load_config(CONFIG_PATH)
print(test)

{'database': {'user': None, 'password': None, 'host': None, 'port': None, 'db': None}}


# AHMED

In [53]:
csv = ROOT_DIR / "data" / "acc_2017.csv"
print(csv)

c:\Users\DELL\Documents\vscode_simplon\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\data\acc_2017.csv


# ROMAIN

In [23]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

API: https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/acci...


In [25]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

EXTRACTION DES DONNÉES
Récupération du nombre total...
Total disponible: 475,911 enregistrements
{'total_count': 475911, 'results': [{'num_acc': '201700009715', 'datetime': '2017-05-28T16:50:00+00:00', 'nom_com': None, 'an': '2017', 'mois': '05', 'jour': '28', 'hrmn': '18:50', 'lum': 'Plein jour', 'agg': 'En agglomération', 'int': '3', 'atm': 'Normale', 'col': 'Deux véhicules – par le coté', 'dep': '13', 'com': '055', 'insee': '13055', 'adr': '6 Av Alexandre  Ansaldi', 'lat': '4333582', 'long': '0539866', 'code_postal': None, 'num': '6', 'coordonnees': {'lon': 2.911777, 'lat': 42.686216}, 'pr': None, 'surf': 'normale', 'v1': None, 'circ': 'Bidirectionnelle', 'vosp': None, 'env1': '00', 'voie': '4', 'larrout': 120, 'v2': None, 'lartpc': 25, 'nbv': 4, 'catr': 'Route Départementale', 'pr1': None, 'plan': 'Partie rectiligne', 'prof': 'Plat', 'infra': None, 'situ': 'Sur chaussée', 'an_nais': ['1998', '1966'], 'sexe': ['Masculin', 'Masculin'], 'actp': ['Se déplaçant', 'Se déplaçant'], 'grav'

In [ ]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""

import sys
from dataclasses import dataclass
from pathlib import Path
import requests



@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


Téléchargement depuis OpenDataSoft...
Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
Téléchargement terminé


In [8]:
# Import du script avec chemin absolu
import sys

# Chemin absolu vers le dossier scripts
scripts_dir = ROOT_DIR / 'etl'

# Ajouter au path Python
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Import
from sauvegarde_csv_api import download_accidents

# Téléchargement
stats = download_accidents(
    output=ROOT_DIR / 'data/accidents_corporels_millesime.csv',
    delimiter=';',
    show_progress=True,
    retries=3,
    limit = 1000,
    where ="an=2017 AND dep='60'"
)

print(f"✅ Téléchargement réussi !")
print(f"   - {stats.record_count:,} accidents")
print(f"   - {stats.column_count} colonnes")
print(f"   - {stats.size_mb:.2f} MB")


🚗 TÉLÉCHARGEMENT DATASET ACCIDENTS CORPORELS
📍 Source: OpenDataSoft
📅 Période: 2012-2019
📄 Fichier: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
🔤 Séparateur: ';'
🔢 Limite: 1000
🔍 Filtre: an=2017 AND dep='60'

⚠️  Module 'tqdm' non installé - pas de barre de progression détaillée
   Installez-le avec: pip install tqdm

🌐 Connexion à OpenDataSoft...
📥 Téléchargement (taille inconnue)
   Téléchargé: 325.3 KB
✅ Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
⏱️  Temps de téléchargement: 0.7 secondes

📊 RÉSUMÉ DU TÉLÉCHARGEMENT
✅ Fichier valide
   Taille: 0.32 MB (333,119 bytes)
   Lignes: 435 (incluant l'en-tête)
   Records: 434 accidents
   Colonnes: 69

📋 Premières colonnes:
   1. num_acc
   2. datetime
   3. nom_com
   4. an
   5. mois
   6. jour
   7. hrmn
   8. lum
   9. agg
   10. int
   ... et 59 autres colonnes

✅ Téléc

# LOUNES

# ZOUBIR